# Diagnostic 1: unique-image-level reassignment (ResNet18)


In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.diagnostic_dataloader import load_multimodal_tensors_shuffled_image
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+resnet18 (diagnostic 1)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'resnet18'
IMG_FEAT_DIM = 512

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_1_resnet'
STABILITY_SUMMARY_PATH = (
    RESULTS_DIR / 'diagnostic1_unique_image_reassignment_resnet18_validation_summary.json'
)


In [4]:
stats = data.load_cgm_stats()

_preview_loader = load_multimodal_tensors_shuffled_image(
    'train', encoder_name=ENCODER_NAME, seed=SEEDS[0]
)
_sample_batch = next(iter(_preview_loader))

print('batch shapes:', [tuple(x.shape) for x in _sample_batch])
print('Diagnostic 1 uses train + validation only; the held-out test set is not loaded.')


batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]
Diagnostic 1 uses train + validation only; the held-out test set is not loaded.


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    # One fixed unique-image reassignment per seed/split.
    train_loader = load_multimodal_tensors_shuffled_image(
        'train', encoder_name=ENCODER_NAME, seed=seed
    )
    val_loader = load_multimodal_tensors_shuffled_image(
        'val', encoder_name=ENCODER_NAME, seed=seed, shuffle=False
    )

    train.set_seed(seed)
    model = MultimodalGRUPredictor(
        img_feat_dim=IMG_FEAT_DIM,
        gru_hidden_size=256,
        gru_num_layers=3,
        gru_dropout=0.2,
    )

    model, history = train.train_model(
        model,
        train_loader,
        val_loader,
        stats['cgm_mean'],
        stats['cgm_std'],
        lr=0.0001,
        weight_decay=1e-05,
        forward_fn=forward_fn,
        checkpoint_dir=(
            EXPERIMENT_DIR / 'checkpoints'
            / f'diagnostic1_unique_image_reassignment_resnet18_seed{seed}'
        ),
        verbose=False,
        progress_bar=True,
        progress_desc=f'seed={seed}',
    )

    n_epochs = len(history['train_loss'])
    val_history = history.get('val_loss', [])
    best_epoch = int(np.argmin(val_history) + 1) if len(val_history) else n_epochs

    y_pred_val, y_true_val = train.predict(
        model, val_loader, forward_fn=forward_fn
    )
    val_results = metrics.evaluate(
        y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std']
    )

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'n_epochs_trained': n_epochs,
        'best_epoch': best_epoch,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_history],
        'val_results': val_results,
    }
    all_results.append(seed_result)

    with open(
        RESULTS_DIR / f'diagnostic1_unique_image_reassignment_resnet18_seed{seed}_results.json',
        'w',
    ) as f:
        json.dump(seed_result, f, indent=2)

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(
        f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
        f"best epoch={best_epoch:>3}  "
        f"validation RMSE @ {HORIZONS} min = {val_rmse}\n"
    )


Test results — CGM+time+resnet18 (diagnostic 1) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  13/50  best epoch=  3  validation RMSE @ [15, 30, 60, 90, 120] min = [11.42, 17.63, 25.34, 29.95, 32.87]
Test results — CGM+time+resnet18 (diagnostic 1) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.52, 17.47, 25.12, 30.03, 33.2]
Test results — CGM+time+resnet18 (diagnostic 1) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


...

seed= 56  stopped at epoch  13/50  best epoch=  3  validation RMSE @ [15, 30, 60, 90, 120] min = [11.15, 17.4, 25.14, 29.9, 32.87]
Test results — CGM+time+resnet18 (diagnostic 1) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.62, 17.66, 25.5, 30.36, 33.44]
Test results — CGM+time+resnet18 (diagnostic 1) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  15/50  best epoch=  5  validation RMSE @ [15, 30, 60, 90, 120] min = [11.0, 17.35, 25.29, 29.99, 33.04]


In [7]:
# Diagnostic summary: validation only.
# The held-out test set remains untouched during this model-development diagnostic.

METRICS = ['rmse', 'mae', 'grmse']
summary = {}

for h in HORIZONS:
    summary[h] = {}
    for metric_name in METRICS:
        values = [r['val_results'][h][metric_name] for r in all_results]
        summary[h][metric_name] = {
            'mean': float(np.mean(values)),
            'sd_across_seeds': float(np.std(values, ddof=1)),
        }

best_epochs = [r['best_epoch'] for r in all_results]
stopping_epochs = [r['n_epochs_trained'] for r in all_results]

final_loss_gaps = []
for r in all_results:
    if r['val_loss_history']:
        final_loss_gaps.append(
            float(r['val_loss_history'][-1] - r['train_loss_history'][-1])
        )

print(f'=== {MODEL_NAME}: validation summary across seeds ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min: "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd_across_seeds']:.2f}   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd_across_seeds']:.2f}   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd_across_seeds']:.2f}"
    )

print(
    f"Best epoch: {np.mean(best_epochs):.2f} +/- {np.std(best_epochs, ddof=1):.2f}; "
    f"stopping epoch: {np.mean(stopping_epochs):.2f} +/- "
    f"{np.std(stopping_epochs, ddof=1):.2f}"
)

if final_loss_gaps:
    print(
        f"Final normalized val-train loss gap: "
        f"{np.mean(final_loss_gaps):.6f} +/- {np.std(final_loss_gaps, ddof=1):.6f}"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump(
        {
            'model': MODEL_NAME,
            'seeds': SEEDS,
            'n_seeds': len(SEEDS),
            'per_horizon_validation_summary': summary,
            'best_epochs': best_epochs,
            'stopping_epochs': stopping_epochs,
            'final_normalized_val_minus_train_loss_gaps': final_loss_gaps,
            'test_set_evaluated': False,
        },
        f,
        indent=2,
    )

print(f'\nsaved per-seed validation results to {RESULTS_DIR}')
print(f'saved validation summary to {STABILITY_SUMMARY_PATH}')


=== CGM+time+resnet18 (diagnostic 1): validation summary across seeds ===
 15-min: RMSE = 11.28 +/- 0.26   MAE = 7.75 +/- 0.22   gRMSE = 12.33 +/- 0.40
 30-min: RMSE = 17.53 +/- 0.12   MAE = 11.84 +/- 0.09   gRMSE = 19.79 +/- 0.39
 60-min: RMSE = 25.40 +/- 0.23   MAE = 16.84 +/- 0.17   gRMSE = 30.04 +/- 0.51
 90-min: RMSE = 30.16 +/- 0.22   MAE = 20.15 +/- 0.15   gRMSE = 36.47 +/- 0.47
120-min: RMSE = 33.11 +/- 0.19   MAE = 22.53 +/- 0.18   gRMSE = 40.42 +/- 0.37
Best epoch: 3.80 +/- 1.99; stopping epoch: 13.80 +/- 1.99
Final normalized val-train loss gap: 0.069701 +/- 0.006262

saved per-seed validation results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_1_resnet
saved validation summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_1_resnet/diagnostic1_unique_image_reassignment_resnet18_validation_summary.json


### Comparation with real images resnet18

In [28]:
REAL_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' /'multimodal' /'results' /'gated_resnet18'
REAL_STUB = 'gated_cgm_time_resnet18'
DIAG_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' /'results' / 'diagnostic_1_resnet'
DIAG_STUB = 'diagnostic1_unique_image_reassignment_resnet18'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
HORIZONS = [15, 30, 60, 90, 120]
EARLY_STOP_PATIENCE = 10  # src/train.py default, used unchanged throughout this dissertation

In [29]:
def load_seed_results(dir_, stub, seeds):
    out = []
    for seed in seeds:
        path = dir_ / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(f'missing {path} -- has this run finished all {len(seeds)} seeds?')
        with open(path) as f:
            out.append(json.load(f))
    return out

real_results = load_seed_results(REAL_DIR, REAL_STUB, SEEDS)
diag_results = load_seed_results(DIAG_DIR, DIAG_STUB, SEEDS)
print(f'loaded {len(real_results)} real-image seeds, {len(diag_results)} diagnostic1 seeds')

loaded 10 real-image seeds, 10 diagnostic1 seeds


In [30]:
# Epoch-count comparison: how fast did each condition stop / (for diagnostic1) hit its best epoch
real_stopping = [r['n_epochs_trained'] for r in real_results]
real_best_inferred = [max(1, n - EARLY_STOP_PATIENCE) for n in real_stopping]
diag_stopping = [r['n_epochs_trained'] for r in diag_results]
diag_best = [r['best_epoch'] for r in diag_results]

print('=== Epoch counts (mean +/- sd across 10 seeds) ===')
print(f"{'':<28}{'stopping epoch':>18}{'best epoch':>18}")
print(f"{'real (correct image)':<28}"
      f"{np.mean(real_stopping):>12.2f}+/-{np.std(real_stopping, ddof=1):<4.2f}"
      f"{np.mean(real_best_inferred):>12.2f}+/-{np.std(real_best_inferred, ddof=1):<4.2f} (inferred)")
print(f"{'diagnostic1 (reassigned)':<28}"
      f"{np.mean(diag_stopping):>12.2f}+/-{np.std(diag_stopping, ddof=1):<4.2f}"
      f"{np.mean(diag_best):>12.2f}+/-{np.std(diag_best, ddof=1):<4.2f}")

diag_loss_gaps = []
for r in diag_results:
    if r.get('val_loss_history') and r.get('train_loss_history'):
        diag_loss_gaps.append(r['val_loss_history'][-1] - r['train_loss_history'][-1])
if diag_loss_gaps:
    print(f"\ndiagnostic1 final (val - train) loss gap: "
          f"{np.mean(diag_loss_gaps):.6f} +/- {np.std(diag_loss_gaps, ddof=1):.6f}")
print("real run's per-epoch loss history was not saved -- no directly-comparable loss-gap number "
      "(see markdown cell above)")

=== Epoch counts (mean +/- sd across 10 seeds) ===
                                stopping epoch        best epoch
real (correct image)               13.40+/-2.27        3.40+/-2.27 (inferred)
diagnostic1 (reassigned)           13.80+/-1.99        3.80+/-1.99

diagnostic1 final (val - train) loss gap: 0.069701 +/- 0.006262
real run's per-epoch loss history was not saved -- no directly-comparable loss-gap number (see markdown cell above)


In [31]:
print('=== Validation RMSE: real (correct image) vs. diagnostic1 (reassigned image) ===')
print(f"{'horizon':>10}{'real':>14}{'diagnostic1':>16}{'delta':>10}")
for h in HORIZONS:
    real_vals = [r['val_results'][str(h)]['rmse'] for r in real_results]
    diag_vals = [r['val_results'][str(h)]['rmse'] for r in diag_results]
    real_mean, diag_mean = np.mean(real_vals), np.mean(diag_vals)
    print(f"{h:>8}min{real_mean:>10.2f}+/-{np.std(real_vals, ddof=1):<4.2f}"
          f"{diag_mean:>12.2f}+/-{np.std(diag_vals, ddof=1):<4.2f}"
          f"{diag_mean - real_mean:>+10.2f}")

print()
print('If diagnostic1 (reassigned image) closely matches real (correct image) on both epoch-count')
print('and RMSE -- the failure has nothing to do with image content, only with adding a branch of')
print('this shape/capacity. A clear gap in either direction means the true image-target')
print('correspondence itself matters.')

=== Validation RMSE: real (correct image) vs. diagnostic1 (reassigned image) ===
   horizon          real     diagnostic1     delta
      15min     11.48+/-0.30       11.28+/-0.26     -0.21
      30min     17.83+/-0.09       17.53+/-0.12     -0.30
      60min     25.86+/-0.21       25.40+/-0.23     -0.46
      90min     30.65+/-0.13       30.16+/-0.22     -0.49
     120min     33.38+/-0.11       33.11+/-0.19     -0.27

If diagnostic1 (reassigned image) closely matches real (correct image) on both epoch-count
and RMSE -- the failure has nothing to do with image content, only with adding a branch of
this shape/capacity. A clear gap in either direction means the true image-target
correspondence itself matters.
